In [1]:
2+2

4

In [2]:
import os
from getpass import getpass

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter GROQ_API_KEY")

if not os.getenv("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass("Enter TAVILY_API_KEY")

if not os.getenv("PINECONE_API_KEY"):
    os.environ["PINECONE_API_KEY"] = getpass("Enter PINECONE_API_KEY")

print("GROQ_API_KEY configured:",bool(os.getenv("GROQ_API_KEY")))
print("TAVILY_API_KEY configured:",bool(os.getenv("TAVILY_API_KEY")))
print("PINECONE_API_KEY configured:",bool(os.getenv("PINECONE_API_KEY")))

GROQ_API_KEY configured: True
TAVILY_API_KEY configured: True
PINECONE_API_KEY configured: True


# Load a document 

In [7]:
from langchain_community.document_loaders import WebBaseLoader

SOURCE_URL = "https://docs.langchain.com/oss/python/langgraph/agentic-rag"

loader = WebBaseLoader(
    web_path=SOURCE_URL,
    requests_kwargs={
        "headers":{
            "User-Agent": "Mozilla/5.0 Agentic-RAG-Industry-Demo"
        }
    }
)

raw_docs = loader.load()

print("Loaded documents:", len(raw_docs))
print("Source:", raw_docs[0].metadata.get("source"))
print("\nPreview:\n")
print(raw_docs[0].page_content[:1500])

Loaded documents: 1
Source: https://docs.langchain.com/oss/python/langgraph/agentic-rag

Preview:






Build a custom RAG agent with LangGraph - Docs by LangChain









Documentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraphBuild a custom RAG agent with LangGraphOverviewManaged Deep AgentsDeep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonLearnTutorialsDeep AgentsData AnalysisDeep ResearchContent BuilderRAGLangChainDeep AgentSemantic searchSQL agentVoice agentMulti-agentSubagents: Personal assistantHandoffs: Customer supportRouter: Knowledge baseSkills: SQL assistantLangGraphCustom RAG agentCustom SQL agentConceptual overviewsLangChain vs. LangGraph vs. Deep AgentsProviders and modelsComponent architectureMemoryContextHuman-in-the-loopGraph A

In [8]:
# Split the documents into chunks

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter  =  RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap = 150,
    add_start_index = True
)

chunks = splitter.split_documents(raw_docs)

print("Toatal chunks :", len(chunks))
print("\nFirst chunk preview:\n")
print(chunks[0].page_content[:900])

Toatal chunks : 29

First chunk preview:

Build a custom RAG agent with LangGraph - Docs by LangChain


In [10]:
## Create Local Embeddings


In [11]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs = {"normalize_embeddings":True}
)

sample_vector = embeddings.embed_query("What is agentic rag?")

print("Embedding dimensions:", len(sample_vector))

e:\Agentic RAG\Enterprise-IT-Support-Agentic-RAG-Copilot\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2108.02it/s]


Embedding dimensions: 384


In [12]:
print(sample_vector)

[-0.06795874238014221, 0.04574918746948242, -0.039988186210393906, -0.024486400187015533, -0.015574632212519646, 0.01199745386838913, 0.07909312099218369, 0.056285448372364044, -0.026718800887465477, 0.03930016607046127, 0.04286826401948929, 0.017323758453130722, 0.023811133578419685, 0.011839802376925945, -0.026094473898410797, 0.05797798931598663, 0.08081162720918655, 0.09711681306362152, -0.03452673926949501, 0.04521506652235985, -0.004475786816328764, 0.04435870796442032, -0.012346603907644749, -0.035983916372060776, -0.04619063064455986, 0.03801551088690758, -0.02439129911363125, 0.008649853989481926, 0.01989157125353813, -0.02670281007885933, 0.029451213777065277, 0.10643841326236725, 0.004398680292069912, -0.04615795239806175, 0.048952508717775345, 0.01707373559474945, -0.07043418288230896, 0.03234690800309181, 0.07115574926137924, 0.067536361515522, 0.04229779168963432, -0.023420335724949837, -0.020442526787519455, -0.05911658704280853, 0.011235571466386318, 0.01741765439510345

In [13]:
# Create the PineCone Vector databased


In [15]:
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
import time

INDEX_NAME = "industry-agentic-rag-kb"
NAMESPACE = "langgraph-agentic-rag"

#connect to pinecone
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

#Create the index only if it does not exist.
existing_indexes = [index_info["name"] for index_info in pc.list_indexes()]

if INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=INDEX_NAME,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        ),
    )

    #Wait until Pinecone reports the new index as ready.
    while not pc.describe_index(INDEX_NAME).status["ready"]:
        time.sleep(1)

print("Pinecone index ready:", INDEX_NAME)

#upload the documents chunks and create the langchain vector store

vectorstore = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name = INDEX_NAME,
    namespace = NAMESPACE
)

retriever = vectorstore.as_retriever(
    search_kwargs = {
        "k":4,
        "namespace": NAMESPACE,
    }
)

print("Pinecone vector database and retriever are ready..")

Pinecone index ready: industry-agentic-rag-kb
Pinecone vector database and retriever are ready..


In [18]:
INDEX_NAME = "industry-agentic-rag-kb"
NAMESPACE = "langgraph-agentic-rag"

#connect to pinecone
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

#load the existing index
index = pc.Index(INDEX_NAME)

#connect existing index with Langchain
vectorstore = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    namespace=NAMESPACE
)

retriever = vectorstore.as_retriever(
    search_kwargs = {
        "k":4,
        "namespace": NAMESPACE,
    }
)

print("Existing Pinecone index loaded..",INDEX_NAME)

Existing Pinecone index loaded.. industry-agentic-rag-kb


In [19]:
## Test Private KB Retrievqal


In [20]:
test_question = "What happens if retieved documents are not relevant in Agentic RAG?"

kb_docs = retriever.invoke(test_question)

for i, doc in enumerate(kb_docs,1):
    print(f"\n--- KB RESULT {i} ---")
    print("Source:", doc.metadata.get("source"))
    print(doc.page_content[:700])


--- KB RESULT 1 ---
Source: https://docs.langchain.com/oss/python/langgraph/agentic-rag
Build a custom RAG agent with LangGraph - Docs by LangChain

--- KB RESULT 2 ---
Source: https://docs.langchain.com/oss/python/langgraph/agentic-rag
APIAdditional resourcesUse docs programmaticallyLangChain AcademyCase studiesGet helpOn this pageBack to topConceptsSetupSet up LangSmithPreprocess documentsCreate a retriever toolGenerate a query or respondGrade documentsRewrite the questionGenerate an answerAssemble the graphRun the agentic RAGSee alsoTutorialsLangGraphBuild a custom RAG agent with LangGraphCopy pageCopy pageBuild a custom retrieval agent with LangGraph that decides when to search a vector store or respond directly.Copy pageCopy pageBuild a retrieval agent with LangGraph that decides when to search a vector store versus answering the user directly.

--- KB RESULT 3 ---
Source: https://docs.langchain.com/oss/python/langgraph/agentic-rag
grader_model = init_chat_model("openai:gpt-5.4-m

In [21]:
## Initialize Groq LLM

In [22]:
from langchain_groq import ChatGroq

llm  = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

print(llm.invoke("Explain AGentic RAG in one sentence.").content)

AGentic RAG is a framework that blends autonomous agentic reasoning with retrieval‑augmented generation, enabling a system to dynamically query external knowledge bases and incorporate the retrieved information into its responses.


In [23]:
## Initialize the Tavily Web Search

In [24]:
from langchain_tavily import TavilySearch

web_search = TavilySearch(
    max_results = 5,
    topic = "general",
    include_answer = True,
    include_raw_content = False
)

print("Tavily search tool ready....")

Tavily search tool ready....


In [ ]:
## Define Structured Decisions